# Question 4

Evolve the table's schema two ways: append a new column using mergeSchema, then change an existing column's type using overwriteSchema; document the difference in what each requires.

In [0]:
%sql 
create  schema if not exists cyntexa_dev.Day_7

In [0]:
data_v1 = [
    (1, "Alice"),
    (2, "Bob")
]

df_v1 = spark.createDataFrame(data_v1, ["id", "name"])

df_v1.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true")\
    .saveAsTable("cyntexa_dev.Day_7.dummy__table")


# --- STEP 1: Append New Data ---

data_v2 = [
    (3, "Charlie", 29),
    (4, "Diana", 34)
]

df_v2 = spark.createDataFrame(data_v2, ["id", "name", "age"])

df_v2.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("cyntexa_dev.Day_7.dummy__table")

In [0]:
# this is code will throw error ,because we   mergeschemra do not work for data mismatch ,to resolve this problem , we need to use overwrite schema but there are few drawbackes or using overwrite schema .

df_v2 = spark.createDataFrame([("5001_", "Evan", 40)], ["id", "name", "age"])
df_v2.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("cyntexa_dev.Day_7.dummy__table")

In [0]:
#  This is code will work because we have changed the schema of the table.

df_v2 = spark.createDataFrame([("_5001", "Evan", 40)], ["id", "name", "age"])
df_v2.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("cyntexa_dev.Day_7.dummy__table")

In [0]:
%sql
select * from cyntexa_dev.Day_7.dummy__table version as of 1


# Delta Lake Schema Evolution: `mergeSchema` vs `overwriteSchema`

## Step-by-Step

### 1. Create the Initial Delta Table

Create an initial Delta table with columns:

* `id INT`
* `name STRING`

### 2. First Schema Evolution — `mergeSchema`

Append new data containing an additional column:

* `id INT`
* `name STRING`
* `age INT`

Use:

```python
.option("mergeSchema", "true").mode("append")
```

This adds the new `age` column to the existing table schema while **preserving the previous data**.

For older records, the newly added `age` column will contain `NULL`.

### 3. Second Schema Evolution — `overwriteSchema`

Now write data where an existing column's data type has changed. For example:

* Previous: `id INT`
* New: `id STRING`

Use:

```python
.option("overwriteSchema", "true").mode("overwrite")
```

This **replaces the existing table schema and data** with the new DataFrame's schema and data.

### 4. Difference Between `mergeSchema` and `overwriteSchema`

| Feature                 | `mergeSchema`                                                               | `overwriteSchema`                                   |
| ----------------------- | --------------------------------------------------------------------------- | --------------------------------------------------- |
| **Write Mode Required** | Usually `append` (can also be used with `overwrite` in supported scenarios) | Must be `overwrite`                                 |
| **Data Retention**      | Preserves existing data; new columns are `NULL` for old records             | Existing data is replaced by the new data           |
| **Type Changes**        | Cannot generally change an existing column's data type                      | Allows redefining existing column data types        |
| **Main Purpose**        | Adding new columns as the dataset evolves                                   | Completely replacing/restructuring the table schema |
| **Example**             | Add `age INT` to `id INT, name STRING`                                      | Change `id INT` to `id STRING`                      |

## Key Difference

* **`mergeSchema`** → **Merge the new schema into the existing schema.** Existing data is retained.
* **`overwriteSchema`** → **Replace the existing schema and table data with the new schema and data.**



# Question 5
 Set up an Autoloader stream ingesting from a folder, then drop 2 more files into the folder and confirm they're picked up automatically. 


In [0]:
%sql
create volume if not exists cyntexa_dev.Day_7.my_volume
    


In [0]:
# 1. Volume Paths (Source ko alag subfolder me rakho)
source_volume = "/Volumes/cyntexa_dev/day_7/my_volume/source_dir/" # <-- YAHAN CHANGE HUA HAI
schema_volume = "/Volumes/cyntexa_dev/day_7/my_volume/schema_sales"
checkpoint_volume = "/Volumes/cyntexa_dev/day_7/my_volume/check_points"

# 2. Read Stream
df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")                            # CSV headers ke liye
    .option("cloudFiles.schemaLocation", schema_volume)
    .load(source_volume)
)


# 3. Write Stream
(df.writeStream
    .trigger(availableNow=True)
    .option("checkpointLocation", checkpoint_volume)
    .toTable("cyntexa_dev.day_7.autoloader_sales_table")  # Unity Catalog me toTable best practice hai
)

In [0]:
%sql
select * from   cyntexa_dev.day_7.autoloader_sales_table;


# Question 6

 Use RESTORE to roll a table back to a version before a bad schema change, and describe what happens to the versions that were created after the point you restored to. 

In [0]:
%sql 
desc history cyntexa_dev.day_7.autoloader_sales_table;
RESTORE TABLE cyntexa_dev.day_7.autoloader_sales_table TO VERSION AS OF 1;
select * from  cyntexa_dev.day_7.autoloader_sales_table ;

No Deletion or Overwriting: Delta Lake does not delete or erase intermediate versions (e.g., Version 2) that contained the unwanted schema or _rescued_data. Delta architecture is strictly append-only and immutable.

New Commit Version Generated: Restoring creates a brand-new commit version (e.g., Version 3). This new version points to the exact dataset and schema state of Version 1.

Audit Trail & Time Travel Preserved: The entire transaction log (_delta_log) remains intact. You can still query or inspect the intermediate bad version using Time Travel: